In [0]:
select * from parquet.`dbfs:/databricks-datasets/credit-card-fraud/data/part-00000-tid-898991165078798880-9c1caa7b-283d-47c4-9be1-aa61587b3675-0-c000.snappy.parquet` limit 5; 

In [0]:
select 
* 
, current_date() as data -- adiciona uma coluna data a data atual no final da tabela 
from parquet.`dbfs:/databricks-datasets/credit-card-fraud/data/part-00000-tid-898991165078798880-9c1caa7b-283d-47c4-9be1-aa61587b3675-0-c000.snappy.parquet`
limit 5; 

In [0]:
-- O comando CREATE OR REPLACE AS é um adicional da CTAS porém com a vantagem que você faz um overwrite na tabela toda vez que toda o comando 
-- Garante a segurança (ACID) nas transações e pode navegar na linha do tempo 

CREATE OR REPLACE TABLE credit_card_fraud_parquet AS 
select * 
, current_date() as data 
from parquet.`dbfs:/databricks-datasets/credit-card-fraud/data/part-00000-tid-898991165078798880-9c1caa7b-283d-47c4-9be1-aa61587b3675-0-c000.snappy.parquet`

In [0]:
SELECT * FROM credit_card_fraud_parquet LIMIT 10

In [0]:
DESCRIBE DETAIL credit_card_fraud_parquet

In [0]:
DESCRIBE EXTENDED credit_card_fraud_parquet

In [0]:
DESCRIBE HISTORY credit_card_fraud_parquet

In [0]:
-- O comando INSERT OVERWRITE sobrescreve a tabela toda vez que rodar, porém não é possível alterar o schema da tabela (exemplo: adicionar uma coluna)

INSERT OVERWRITE credit_card_fraud_parquet
select * 
, current_date() as data 
from parquet.`dbfs:/databricks-datasets/credit-card-fraud/data/part-00000-tid-898991165078798880-9c1caa7b-283d-47c4-9be1-aa61587b3675-0-c000.snappy.parquet`




In [0]:
-- Já o comando INSERT INTO funciona como um append na tabela (também não é possível adicionar colunas novas)
INSERT INTO credit_card_fraud_parquet
select * 
, current_date() as data 
from parquet.`dbfs:/databricks-datasets/credit-card-fraud/data/part-00000-tid-898991165078798880-9c1caa7b-283d-47c4-9be1-aa61587b3675-0-c000.snappy.parquet`


In [0]:

DROP TABLE IF EXISTS credit_card_fraud_parquet;


Selecionando Colunas especificas

In [0]:
CREATE OR REPLACE TABLE credit_card_fraud_colunas AS 
select 
  `time` -- utilize crase quando a coluna tiver nome de função
  ,amountRange
from parquet.`dbfs:/databricks-datasets/credit-card-fraud/data/part-00000-tid-898991165078798880-9c1caa7b-283d-47c4-9be1-aa61587b3675-0-c000.snappy.parquet`

In [0]:
-- Como mencionado acima, quando tentamos fazer o insert de dados com uma coluna adicional, a execução irá falhar com o seguinte erro:
-- [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.

insert into credit_card_fraud_colunas
select 
  `time`
  ,amountRange
  ,current_timestamp() as DateTime
from parquet.`dbfs:/databricks-datasets/credit-card-fraud/data/part-00000-tid-898991165078798880-9c1caa7b-283d-47c4-9be1-aa61587b3675-0-c000.snappy.parquet`


In [0]:
-- uma forma de ajustar isso é alterar o schema da tabela, adicionando a coluna desejada 
-- torne a rodar a célula acima 

ALTER TABLE credit_card_fraud_colunas
ADD COLUMNS (DateTime TIMESTAMP);

Note que como a coluna foi adiciona após a criação da tabela, os dados antigos serão preenchido com null na nova coluna 

In [0]:
select DateTime, count(*) as QTD from credit_card_fraud_colunas
group by DateTime; 

In [0]:
drop  table if exists credit_card_fraud_colunas

Criando Views com origem de arquivos 



## Comparação de Views 
Uma exibição ou View é o resultado de uma consulta em uma ou mais tabelas e exibições

### Views Armazenadas (Stored Views)
- **Persistidas no Banco de Dados**: Views armazenadas são salvas no banco de dados e persistem entre sessões.
- **Removidas**: Essas views só podem ser removidas usando o comando `DROP VIEW`.
- **Criação**: Use a instrução `CREATE VIEW` para criar views armazenadas.

### Views Temporárias (Temp Views)
- **Escopo da Sessão**: Views temporárias estão disponíveis apenas na sessão atual.
- **Removidas**: Essas views são removidas quando a sessão termina.
- **Criação**: Use a instrução `CREATE TEMP VIEW` para criar views temporárias.

### Views Temporárias Globais (Global Temp Views)
- **Escopo do Cluster**: Views temporárias globais estão disponíveis em todas as sessões no mesmo cluster.
- **Removidas**: Essas views são removidas quando o cluster é reiniciado.
- **Criação**: Use a instrução `CREATE GLOBAL TEMP VIEW` para criar views temporárias globais.

In [0]:
-- criando view 
CREATE OR REPLACE VIEW credit_card_fraud_view AS
select * from parquet.`dbfs:/databricks-datasets/credit-card-fraud/data/part-00000-tid-898991165078798880-9c1caa7b-283d-47c4-9be1-aa61587b3675-0-c000.snappy.parquet` 



In [0]:
SELECT * FROM credit_card_fraud_view LIMIT 10

In [0]:
CREATE OR REPLACE TEMP VIEW credit_card_fraud_vw_temp AS
select * from parquet.`dbfs:/databricks-datasets/credit-card-fraud/data/part-00000-tid-898991165078798880-9c1caa7b-283d-47c4-9be1-aa61587b3675-0-c000.snappy.parquet` 

In [0]:
SELECT * FROM credit_card_fraud_vw_temp LIMIT 10

In [0]:
-- Não é suportada em serveless

CREATE OR REPLACE GLOBAL TEMP VIEW credit_card_fraud_vw_temp_global
AS
select * from parquet.`dbfs:/databricks-datasets/credit-card-fraud/data/part-00000-tid-898991165078798880-9c1caa7b-283d-47c4-9be1-aa61587b3675-0-c000.snappy.parquet` 

In [0]:
show tables

In [0]:
show tables in global_temp